<a href="https://colab.research.google.com/github/nikhil1770/text2sql-qlora-rbac/blob/main/00_setup_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 — Setup Test
**Project:** text2sql-qlora-rbac
**Goal:** Verify the full training stack works before any real work:
1. GPU (T4) available
2. Unsloth installs and imports
3. Hugging Face token + Llama gated access working
4. Google Drive mounts (for checkpoints)

If all 4 pass, the environment is ready for Phase P1.

## Check 1 — GPU
We need the free Tesla T4 (~15 GB VRAM). QLoRA is designed to fit an 8B model
into exactly this card. If this shows no GPU: Runtime → Change runtime type → T4 GPU.

In [6]:
!nvidia-smi

Thu Jul  2 21:14:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             28W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Check 2 — Install & import Unsloth
Unsloth = optimized fine-tuning library (custom kernels, ~2x faster, less VRAM).
Colab environments are temporary, so this install cell will be repeated at the
top of every GPU notebook. Takes a few minutes. Warnings are normal; only red
ERROR lines matter.

In [2]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

In [3]:
from unsloth import FastLanguageModel
import torch
print("unsloth OK")
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth OK
torch: 2.10.0+cu128 | CUDA available: True


## Check 3 — Hugging Face login + Llama gated access
Token is stored in Colab Secrets (never hardcoded in the notebook — it would
end up public on GitHub!). We verify:
(a) the token authenticates, and
(b) our account has been granted access to the gated meta-llama/Llama-3.1-8B-Instruct repo.
We only check file access here — no 16 GB download in a setup test.

In [4]:
from google.colab import userdata
from huggingface_hub import login, whoami, auth_check

login(token=userdata.get('HF_TOKEN'))
print("Logged in as:", whoami()["name"])

auth_check("meta-llama/Llama-3.1-8B-Instruct")
print("Llama-3.1-8B-Instruct: access GRANTED ✅")

Logged in as: nikhil1772000
Llama-3.1-8B-Instruct: access GRANTED ✅


## Check 4 — Google Drive mount
Colab sessions are temporary; anything not saved to Drive is lost on disconnect.
During training (Phase P3), checkpoints save to Drive every N steps, so a
disconnect costs minutes, not hours. Here we verify the mount works and the
checkpoint folder exists.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
ckpt_dir = '/content/drive/MyDrive/text2sql_project/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

# write-read test
with open(f'{ckpt_dir}/_setup_test.txt', 'w') as f:
    f.write('setup test ok')
print(open(f'{ckpt_dir}/_setup_test.txt').read())
print("Drive checkpoint folder ready ✅")

Mounted at /content/drive
setup test ok
Drive checkpoint folder ready ✅
